In [1]:
import os
import glob
import re
import time
import code
import numpy as np
import pandas as pd
import anndata
import pyfaidx
import pybedtools
from pybedtools import BedTool
import anndata as ad
import scipy.sparse as sp_sparse
from scipy import sparse
import pickle
import scanpy as sc
import networkx as nx
from matplotlib import rcParams
import sys
import scipy.io as sio

import os
import gzip
import shutil


sys.path.append("/home/wuyan/dygmamba_project/model/dygmamba/src")




from pdata.data_read import read_rna_from_txt, read_atac_featurecounts, anndata_to_bed
from pdata.data_preprocess import filter_atac_base

from pdata.data_preprocess import calculate_rp_distance, calculate_peak_peak_rp_distance
from pdata.data_preprocess import analyze_score_distribution
from pdata.data_preprocess import rna_preprocess, atac_preprocess
from pdata.data_preprocess import dataframe_to_anndata_sparse
from pdata.data_preprocess import find_zero_sum_elements



def check_gtf_library():
    """诊断 read_gtf 来自哪个库，行为不同"""
    try:
        from pyranges import read_gtf
        print("read_gtf 来自 pyranges — 支持直接读 .gtf.gz")
        return 'pyranges'
    except ImportError:
        pass
    try:
        from gtfparse import read_gtf
        print("read_gtf 来自 gtfparse — 不支持 .gtf.gz，需要先解压")
        return 'gtfparse'
    except ImportError:
        pass
    raise ImportError("找不到 read_gtf，请安装: pip install pyranges 或 pip install gtfparse")

def prepare_gtf(gtf_path: str) -> str:
    """
    处理各种 GTF 格式，返回可直接读取的文件路径。
    支持:
      - .gtf          → 直接返回
      - .gtf.gz       → 解压到同目录，返回解压路径
      - .tar.gz       → 提取内部 genes/genes.gtf，返回路径
    """
    if not os.path.exists(gtf_path):
        raise FileNotFoundError(f"找不到文件: {gtf_path}")

    # 已经是普通 GTF
    if gtf_path.endswith('.gtf') and not gtf_path.endswith('.tar.gz'):
        print(f"GTF 文件就绪: {gtf_path}")
        return gtf_path

    # gzip 压缩的 GTF（.gtf.gz）
    if gtf_path.endswith('.gtf.gz'):
        out_path = gtf_path[:-3]  # 去掉 .gz
        if not os.path.exists(out_path):
            print(f"解压 {gtf_path} → {out_path} ...")
            with gzip.open(gtf_path, 'rb') as f_in, \
                 open(out_path, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
            print(f"解压完成: {out_path}")
        else:
            print(f"已存在解压文件: {out_path}")
        return out_path

    # tar.gz（10x 参考包）
    if gtf_path.endswith('.tar.gz'):
        import tarfile
        extract_dir = os.path.dirname(gtf_path)
        # 10x 包内 GTF 的固定路径
        inner_path = None
        print(f"扫描 tar 包内容: {gtf_path}")
        with tarfile.open(gtf_path, 'r:gz') as tar:
            for member in tar.getmembers():
                if member.name.endswith('genes/genes.gtf'):
                    inner_path = member.name
                    break
        if inner_path is None:
            raise ValueError(f"在 {gtf_path} 中找不到 genes/genes.gtf")

        out_path = os.path.join(extract_dir, inner_path)
        if not os.path.exists(out_path):
            print(f"从 tar 包提取: {inner_path}")
            with tarfile.open(gtf_path, 'r:gz') as tar:
                tar.extract(inner_path, path=extract_dir)
            print(f"提取完成: {out_path}")
        else:
            print(f"已存在提取文件: {out_path}")
        return out_path

    raise ValueError(f"不支持的文件格式: {gtf_path}")


def read_genenotation(gtf_file_path: str):
    try:
        ready_path = prepare_gtf(gtf_file_path)
    except Exception as e:
        print(f"Error 准备 GTF 文件: {e}")
        return None

    try:
        from pyranges import read_gtf
        gr = read_gtf(ready_path)

        # 兼容新旧版本 pyranges
        if hasattr(gr, 'as_df'):
            gtf_df = gr.as_df()          # 新版 pyranges >= 0.0.120
        elif hasattr(gr, 'to_pandas'):
            gtf_df = gr.to_pandas()      # 旧版
        elif hasattr(gr, 'df'):
            gtf_df = gr.df              # 某些中间版本
        else:
            # 终极兜底：直接转 DataFrame
            import pandas as pd
            gtf_df = pd.DataFrame(gr)

        print(f"GTF 读取成功: {gtf_df.shape[0]} 行")
        print(f"列名: {list(gtf_df.columns[:8])}")
        return gtf_df

    except Exception as e:
        print(f"Error 读取 GTF: {e}")
        return None


In [2]:
data_root = "/home/wuyan/dygmamba_project/NewRealPlan/case2/data/AD/process/"
output_path =  data_root + 'model1_CRND8_Microglia' + '/process/'

gtf_file_path = "/home/wuyan/dygmamba_project/NewRealPlan/case2/data/refdata-cellranger-arc-mm10-2020-A-2.0.0/genes/genes.gtf.gz" 

gtf_df = read_genenotation(gtf_file_path)


已存在解压文件: /home/wuyan/dygmamba_project/NewRealPlan/case2/data/refdata-cellranger-arc-mm10-2020-A-2.0.0/genes/genes.gtf
GTF 读取成功: 1780455 行
列名: ['Chromosome', 'Source', 'Feature', 'Start', 'End', 'Score', 'Strand', 'Frame']


In [3]:
gtf_df

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,gene_version,...,transcript_name,transcript_support_level,havana_transcript,exon_number,exon_id,exon_version,protein_id,tag,ccdsid,ont
0,GL456210.1,ENSEMBL,gene,123791,124928,.,+,.,ENSMUSG00000079192,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,GL456210.1,ENSEMBL,transcript,123791,124928,.,+,.,ENSMUSG00000079192,2,...,AC125149.1-201,5,NaN,NaN,NaN,NaN,ENSMUSP00000106995.2,basic,NaN,NaN
2,GL456210.1,ENSEMBL,exon,123791,123905,.,+,.,ENSMUSG00000079192,2,...,AC125149.1-201,5,NaN,1,ENSMUSE00000902010,1,ENSMUSP00000106995.2,basic,NaN,NaN
3,GL456210.1,ENSEMBL,CDS,123791,123905,.,+,0,ENSMUSG00000079192,2,...,AC125149.1-201,5,NaN,1,ENSMUSE00000902010,1,ENSMUSP00000106995.2,basic,NaN,NaN
4,GL456210.1,ENSEMBL,start_codon,123791,123794,.,+,0,ENSMUSG00000079192,2,...,AC125149.1-201,5,NaN,1,ENSMUSE00000902010,1,ENSMUSP00000106995.2,basic,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1780450,chrY,ENSEMBL,exon,90838868,90839177,.,-,.,ENSMUSG00000096850,1,...,Gm21748-201,NA,NaN,1,ENSMUSE00001062768,1,ENSMUSP00000137361.1,appris_principal_1,NaN,NaN
1780451,chrY,ENSEMBL,CDS,90838871,90839177,.,-,0,ENSMUSG00000096850,1,...,Gm21748-201,NA,NaN,1,ENSMUSE00001062768,1,ENSMUSP00000137361.1,appris_principal_1,NaN,NaN
1780452,chrY,ENSEMBL,start_codon,90839174,90839177,.,-,0,ENSMUSG00000096850,1,...,Gm21748-201,NA,NaN,1,ENSMUSE00001062768,1,ENSMUSP00000137361.1,appris_principal_1,NaN,NaN
1780453,chrY,ENSEMBL,stop_codon,90838868,90838871,.,-,0,ENSMUSG00000096850,1,...,Gm21748-201,NA,NaN,1,ENSMUSE00001062768,1,ENSMUSP00000137361.1,appris_principal_1,NaN,NaN


In [4]:
gtf_df.columns

Index(['Chromosome', 'Source', 'Feature', 'Start', 'End', 'Score', 'Strand',
       'Frame', 'gene_id', 'gene_version', 'gene_type', 'gene_name', 'level',
       'mgi_id', 'havana_gene', 'transcript_id', 'transcript_version',
       'transcript_type', 'transcript_name', 'transcript_support_level',
       'havana_transcript', 'exon_number', 'exon_id', 'exon_version',
       'protein_id', 'tag', 'ccdsid', 'ont'],
      dtype='object')

In [9]:
gtf_df2 = gtf_df[gtf_df['Feature']== 'gene'].copy()
gtf_df2

,Chromosome,Source,Feature,Start,End,Score,Strand,Frame,gene_id,gene_version,...,transcript_name,transcript_support_level,havana_transcript,exon_number,exon_id,exon_version,protein_id,tag,ccdsid,ont
0,GL456210.1,ENSEMBL,gene,123791,124928,.,+,.,ENSMUSG00000079192,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,GL456210.1,ENSEMBL,gene,147791,149707,.,+,.,ENSMUSG00000094799,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,GL456210.1,ENSEMBL,gene,9123,58882,.,-,.,ENSMUSG00000079800,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26,GL456210.1,ENSEMBL,gene,108389,110303,.,-,.,ENSMUSG00000095092,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35,GL456210.1,ENSEMBL,gene,135394,136519,.,-,.,ENSMUSG00000079794,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1780416,chrY,HAVANA,gene,89927119,89935013,.,-,.,ENSMUSG00000102011,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1780421,chrY,HAVANA,gene,90603500,90605864,.,-,.,ENSMUSG00000099619,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1780431,chrY,HAVANA,gene,90665345,90667625,.,-,.,ENSMUSG00000099399,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1780441,chrY,HAVANA,gene,90752426,90755467,.,-,.,ENSMUSG00000095366,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
